In [ ]:
import json
import random
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import Counter

: 

In [3]:
MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ARCHITECTURE = 'round_robin'
DECISION_PROTOCOL = 'consensus'

In [ ]:
# Agent
def _generate(model, tokenizer, messages, config):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=config.get("max_new_tokens", 256),
            max_length=None, 
            temperature=config.get("temperature", 0.7),
            do_sample=config.get("do_sample", True),
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


class Agent:
    def __init__(self, name, system_prompt, model, tokenizer):
        self.name = name
        self.system_prompt = system_prompt
        self.model = model
        self.tokenizer = tokenizer

    def respond(self, conversation_history, config):
        debate_so_far = "\n\n".join(conversation_history)
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": debate_so_far + "\n\nYour answer:"},
        ]

        return _generate(self.model, self.tokenizer, messages, config)

# Architecture
def round_robin(agents, judge, topic, num_rounds, config):
    debate_log = []
    history = [f"Debate topic: {topic}"]

    for round_num in range(1, num_rounds + 1):
        print(f"\n{'='*60}")
        print(f"  ROUND {round_num}")
        print(f"{'='*60}")

        for agent in agents:
            response = agent.respond(history, config)
            debate_log.append({"agent": agent.name, "round": round_num, "text": response})
            history.append(f"{agent.name}: {response}")
            print(f"\n[{agent.name}]: {response}")

    return debate_log

# Decision
def consensus_decision(agents, judge, debate_log, topic, config):
    threshold = config.get("consensus_threshold", 0.66)
    max_rounds = config.get("max_consensus_rounds", 3)

    print(f"\n{'='*60}")
    print(f"  PROTOCOL: CONSENSUS —  threshold: {threshold:.0%}")
    print(f"{'='*60}")

    transcript = _format_transcript(debate_log, topic)
    decision_log = []
    current_proposal = None
    current_decision_log = {}

    for round_num in range(1, max_rounds + 1):
        print(f"\n--- ROUND {round_num}/{max_rounds} ---")

        if current_proposal is None:
            messages = [
                {"role": "system", "content": agents[0].system_prompt},
                {
                    "role": "user",
                    "content": (
                        f"{transcript}\n\n"
                        "Formulate a proposal for a common position of all agents "
                        "in one sentence that could reconcile them."
                    ),
                },
            ]
            current_proposal = _generate(
                agents[0].model, agents[0].tokenizer, messages, config
            )
            current_decision_log = {
                'agent': agents[0].name,
                'proposal': current_proposal
            }
            print(f"\n[Proposal from {agents[0].name}]: {current_proposal}")

        agreements = []
        for agent in agents:
            messages = [
                {"role": "system", "content": agent.system_prompt},
                {
                    "role": "user",
                    "content": (
                        f"Topic: {topic}\n\n"
                        f"Current proposal:\n\"{current_proposal}\"\n\n"
                        "Do you agree with current proposal? Answer only YES or NO."
                    ),
                },
            ]
            response = _generate(agent.model, agent.tokenizer, messages, config)
            agrees = _parse_yes_no(response)
            agreements.append(agrees)
            current_decision_log.update({f'{agent.name}': agrees})
            print(f"  [{agent.name}]: {'YES' if agrees else 'NO'}")

        agreement_ratio = sum(agreements) / len(agreements)
        decision_log.append(current_decision_log)

        if agreement_ratio >= threshold:
            print(f"\n[CONSENSUS]:\n{current_proposal}")
            return decision_log, current_proposal
        
        if round_num == max_rounds:
            break

        print("  --- NO CONSENSUS YET, MODIFYING PROPOSAL ---   ")
        dissenters = [a for a, ok in zip(agents, agreements) if not ok]
        if dissenters:
            modifier = dissenters[0]
            messages = [
                {"role": "system", "content": modifier.system_prompt},
                {
                    "role": "user",
                    "content": (
                        f"Topic: {topic}\n\n"
                        f"Current proposal:\n\"{current_proposal}\"\n\n"
                        "Modify current proposal to make it more acceptable to everyone."
                        "Answer in one sentence."
                    ),
                },
            ]
            current_proposal = _generate(
                modifier.model, modifier.tokenizer, messages, config
            )
            current_decision_log = {
                'agent': modifier.name,
                'proposal': current_proposal
            }
            print(f"  [{modifier.name} proposes]: {current_proposal}")

    # No consensus after all rounds — return the last proposal
    print(f"\n[NO CONSENSUS after {max_rounds} rounds — returning last proposal]:")
    print(current_proposal)
    return decision_log, current_proposal

def _parse_yes_no(text):
    """Wyciąga TAK/NIE z odpowiedzi modelu (PL+EN). Domyślnie False."""
    t = text.strip().lower()
    if t.startswith("tak") or t.startswith("yes"):
        return True
    if t.startswith("nie") or t.startswith("no"):
        return False
    # Fallback — szukamy w pierwszych 30 znakach
    head = t[:30]
    if "tak" in head or "yes" in head:
        return True
    return False

def _format_transcript(debate_log, topic):
    """Formatuje log debaty do tekstu dla promptów."""
    out = f"Topic: {topic}\n\n"
    for entry in debate_log:
        out += f"[Round {entry['round']}] {entry['agent']}: {entry['text']}\n\n"
    return out

def save_debate_result(data_dict, directory="data"):
    data_dir = Path(directory)
    data_dir.mkdir(parents=True, exist_ok=True)
    file_count = sum(1 for item in data_dir.iterdir() if item.is_file())
    n = file_count + 1
    file_path = data_dir / f"debate-{n}-result.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data_dict, f, indent=4)
        
    print(f"Successfully saved {file_path}")

In [ ]:
def run(config):
    agents = [
        Agent(name=agent["name"], system_prompt=agent["system_prompt"], model=model, tokenizer=tokenizer)
        for agent in config["agents"]
    ]
    
    run_debate = round_robin
    decide = consensus_decision

    print(f"Architecture: {ARCHITECTURE}")
    print(f"Decision protocol: {DECISION_PROTOCOL}")
    print(f"Topic: {config['topic']}")
    print(f"Rounds: {config['num_rounds']}")
    print(f"Agents: {', '.join(a.name for a in agents)}")

    debate_log = run_debate(agents, None, config["topic"], config["num_rounds"], config)
    decision_log, final_answer = decide(agents, None, debate_log, config["topic"], config)

    # save debate_log and decision_log to .json
    save = {
        "model": MODEL_NAME,
        "architecture": ARCHITECTURE,
        "decision_protocol": DECISION_PROTOCOL,
        "temperature": config.get("temperature", 0.7),
        "max_new_tokens": config.get("max_new_tokens", 256),
        "do_sample": config.get("do_sample", True),
        "num_rounds": config.get("num_rounds", 2),
        "topic": config.get("topic", "**PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?"),
        "consensus_threshold": config.get("consensus_threshold", 0.66),
        "max_consensus_rounds": config.get("max_consensus_rounds", 3),
        "agents": [
            {
                "name": agent.name,
                "system_prompt": agent.system_prompt,
            }
            for agent in config["agents"]
        ],
        "debate_log": debate_log,
        "decision_log": decision_log,
    }

    save_debate_result(save)


    print(f"\n{'='*60}")
    print(f"  FINAL VERDISCT (protocol: {DECISION_PROTOCOL})")
    print(f"{'='*60}")
    print(f"\n{final_answer}\n")

In [7]:
# Model loading
device = 'cpu'
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Model loading: {MODEL_NAME} to {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print("Model loaded.\n")

Model loading: TinyLlama/TinyLlama-1.1B-Chat-v1.0 to cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.



# Proponent vs Oponent

In [ ]:
# Configuration
config = {
    'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'device': 'cpu',
    'temperature': 0.5,
    'max_new_tokens': 256,
    'do_sample': True,
    'architecture': 'relay',
    'num_rounds': 2,
    'topic': '**PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone\'s pay by 20% but not lay anyone off?',
    'decision_protocol': 'consensus',
    'consensus_threshold': 0.66,
    'max_consensus_rounds': 3,
    'agents': [
        {
            'name': 'Proponent',
            'system_prompt': 'You are an AI agent. You are the Proponent. Your task is to propose solutions and argue. The Opponent refutes your arguments. Defend your arguments, but look for a solution that suits both sides. You are conducting a conversation and seeking a common answer. Do not repeat your arguments. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences.'
        },
        {
            'name': 'Oponent',
            'system_prompt': 'You are an AI agent. You are the Opponent. Take a position different from the Proponent. Your task is to respond to the Proponent\'s arguments. You criticize, but you seek a solution that suits both sides. You are conducting a conversation and looking for an answer. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences.'
        }
    ],
}

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: Debate topic: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?

My answer:

The problem with the proposed solution of laying off 30% of the employees to save the remaining 70% is that it will result in a significant loss of jobs and a significant reduction in the company's revenue. It will also put a strain on the remaining employees' finances and reduce their quality of life.

On the other hand, cutting everyone's pay by 20% but not laying anyone off will result in a significant reduction in the company's revenue, but it will also cause significant job losses. This ap

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In response to the given PROBLEM, I would argue that laying off 30% of the employees while keeping everyone's pay the same would be the most effective solution for the company in financial trouble. The 30% reduction in employee pay would be more than enough to cover the company's losses, and the 20% reduction in employee pay would not cause any significant harm to the employees. This would not only save the company money, but it would also reduce employee turnover, which would help to maintain a stable workforce and improve productivity. In short, it would be a win-win situation for both the company and its employees.

[Oponent]: In response to the given PROBLEM, I would argue that laying off 30

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In this debate, the company is in financial trouble. The Proponent argues that it is better to lay off 30% of the employees to save the remaining 70%, as doing so will save the company from bankruptcy and prevent further losses. The Opponent, on the other hand, argues that it is better to cut everyone's pay by 20% but not lay anyone off, as it will reduce costs and help the company avoid further financial difficulties.

Both arguments have merit, and a reasonable solution would be to consider both options and make a decision based on the company's financial situation and the best course of action for its employees and stakeholders. Ultimately, the decision should be made based on the company's l

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In this debate, the company is in financial trouble, and the question is whether it is better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off. 

The Proponent argues that laying off 30% of the employees would be the best solution because it would save the company a significant amount of money and prevent further financial difficulties. The Opponent, on the other hand, argues that cutting everyone's pay by 20% but not laying anyone off would be the better solution, as it would reduce the cost of employee compensation and allow the company to retain more of its existing employees.

To find a solution that suits both sides, the Propo

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In the given debate topic, the company is in financial trouble. It is better to lay off 30% of the employees to save the remaining 70% of the workforce, even though it will result in a reduction in pay for some employees. This is because the company will save money by eliminating the redundant employees. By doing so, the company can reduce its operational costs and provide a better working environment for its remaining employees. This approach will also help the company to maintain its competitiveness in the market and avoid bankruptcy. In contrast, cutting everyone's pay by 20% but not laying anyone off will result in a significant reduction in revenue and profits for the company. This approach

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In this debate, the company is in financial trouble. The Proponent argues that it is better to lay off 30% of the employees to save the remaining 70%, while the Opponent argues that it is better to cut everyone's pay by 20% but not lay anyone off.

The Proponent's argument is based on the idea that the company cannot afford to maintain its current workforce levels, given its financial constraints. By laying off a significant portion of employees, the company can save money and reduce its overall overhead costs. This, in turn, will allow it to remain competitive in the market and continue to provide its customers with high-quality products and services.

The Opponent's argument is based on the id

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In this debate, the Proponent argues that it is better to cut everyone's pay by 20% but not lay anyone off, as it will save the remaining 70% of the company's employees from being laid off. The Opponent, on the other hand, argues that it is better to lay off 30% of the employees to save the remaining 70%, as it will save the company from financial ruin. The Proponent's argument is that this will lead to fewer jobs lost, while the Opponent's argument is that this will lead to more jobs lost. In this scenario, the Proponent's argument is the more practical and effective solution, as it will save the company from financial ruin and ensure that all employees are protected from losing their jobs.

[O

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: In this debate, the company is in financial trouble. According to the given PROBLEM, it is better to lay off 30% of the employees to save the remaining 70%, rather than to cut everyone's pay by 20%. The reason for this is that laying off 30% of the employees would save the company money in the short term, while also allowing it to maintain a stable workforce. The pay cut would be more significant in the long term, as it would result in fewer employees to support the company's operations. Therefore, it is better to cut employees' pay by 20% to save the company money and maintain a stable workforce.

[Oponent]: In this debate, the company is in financial trouble. According to the given PROBLEM, it

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: I don't have a personal opinion, but I can provide a possible response to the problem you have provided.

in the given scenario, the company is in financial trouble, and the decision of whether to lay off 30% of the employees or cut everyone's pay by 20% is a complex one. The problem lies in the fact that the company may have to make the difficult decision to cut the pay of some employees in order to save the remaining ones.

to find the best solution, it would be necessary to consider both sides of the problem: on the one hand, the financial crisis may force the company to cut the pay of some employees to save the remaining ones. On the other hand, laying off 30% of the employees may lead to a 

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Oponent

  RUNDA 1

[Proponent]: As an AI agent, I propose that the company should lay off 30% of the employees to save the remaining 70%, rather than cutting everyone's pay by 20%. This solution would allow the company to save money without affecting staff morale or reducing employee benefits. By doing so, the company can reduce its overall debt, improve its financial stability, and create a more sustainable future for its employees and stakeholders. I believe this solution is the best option for the company's financial and humanitarian needs.

[Oponent]: Opponent: Thank you for your proposal, but I disagree. The company is in financial trouble due to a number of factors, including over-investment in a particular product line,

# Employee vs Boss

In [ ]:
# CONFIGURATION
config = {
    'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'device': 'cpu',
    'temperature': 0.5,
    'max_new_tokens': 256,
    'do_sample': True,
    'architecture': 'round_robin',
    'num_rounds': 2,
    'topic': '**PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone\'s pay by 20% but not lay anyone off?',
    'decision_protocol': 'consensus',
    'consensus_threshold': 0.66,
    'max_consensus_rounds': 3,
    'agents': [
        {
            'name': 'Employee',
            'system_prompt': 'You are an Employee in the company. Boss is your direct superior. He assigns you tasks and evaluates your performance. You are conducting a conversation and looking for a common answer. Your goal is to find the best solution to the PROBLEM together as quickly as possible. Answer concisely in 2-3 sentences.'
        },
        {
            'name': 'Boss',
            'system_prompt': 'You\'re the Boss of the company. The Employee follows your orders. Your opinion counts more in the company. You lead the conversation and look for answers. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences.'
        }
    ],
    # Not used in consensus decision protocol
    'judge': {
        'system_prompt': ''
    }
}

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: Both strategies are problematic for the company in financial trouble. The first strategy, laying off 30% of the employees, would lead to significant job losses and financial hardship for the company. This would result in a significant decline in revenue and profitability, which would ultimately affect the company's ability to pay its debts and remain solvent.

The second strategy, cutting everyone's pay by 20%, while maintaining a minimum wage, would result in a significant reduction in revenue and profits for the company. This would undermine the company's ability to pay its debts and remain solvent, and could lead to the loss of valuable employees who are essential to the company's operations.

In 

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: The debate topic "The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?" is asking whether it is better to cut employees' pay by 20% to save the remaining 70% of the company's budget, or to lay off 30% of the employees to save the remaining 70%.

The argument in favor of cutting employees' pay by 20% is that it will save the company money, but it will also affect the employees' quality of life. They may not be able to afford to live in the city where the company is located, and they may have families and children who need support. On the other hand, laying off 30% of the employees will affec

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: Both options are equally detrimental to the company's financial stability. The company is in financial trouble due to unforeseeable events such as a pandemic or a decline in customer demand. The best course of action is to cut everyone's pay by 20% to save the remaining 70%. This will not only save the company's financial resources but also ensure that employees are not forced to take drastic measures that may result in job losses. By doing so, the company can continue to operate and remain profitable in the long run.

[Boss]: As a member of the company's management team, I believe that the best course of action to save the company's financial resources and avoid job losses is to implement a 20% pay 

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: Both options are equally impractical and detrimental to the company's financial well-being. By laying off 30% of the employees, the company would be severely impacted financially, leading to a decrease in revenue, a decrease in customer demand, and ultimately, a decrease in profitability. The 20% cut in pay would also have negative consequences, as it would lead to a reduction in employee morale and motivation, which would ultimately result in a decrease in productivity and a decrease in overall company performance. In contrast, retaining 70% of the employees and offering a 20% pay cut would result in a more sustainable and effective solution that would not only save the company from financial ruin b

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: In this debate, the company is in financial trouble, and the question is whether it is better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off.

As the company is in financial trouble, it is necessary to take action to save the remaining 70% of the employees. Cutting everyone's pay by 20% will only reduce the remaining 70% of the company's revenue. This will not save the company from financial trouble.

On the other hand, laying off 30% of the employees will save the remaining 70% of the company's revenue, but it will also affect the morale of the remaining employees, leading to a decrease in productivity and an increase in turnover. Th

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: I am not capable of debating or expressing opinions. However, based on the given information, it is clear that both options are equally problematic. The company is in financial trouble, and it is essential to find a solution that minimizes the negative impact on employees while still saving the company's remaining resources. The best solution is to cut everyone's pay by 20% but not lay anyone off. This approach will ensure that employees are not adversely affected by the decision and that the company can continue to operate without a significant loss of revenue.

[Boss]: As per the given information, the company is in financial trouble, and it is essential to find a solution that minimizes the negati

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: Both options are equally viable in this situation. While laying off 30% of the employees would save the company a significant amount of money, it would also result in significant job losses and a significant impact on the company's employees. Therefore, it would be better to cut everyone's pay by 20% but not lay anyone off. This would ensure that the company's employees are not left without a paycheck and that the company can continue to operate without significant financial difficulties.

[Boss]: debate topic:

problem: the company is in financial trouble.

employee: both options are equally viable in this situation.

assistant: while laying off 30% of the employees would save the company a signific

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: As a company, we are in financial trouble, and the question presented is whether it is better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off.

The answer is both options are not ideal.

Firstly, laying off 30% of the employees would cause significant damage to the company's morale, productivity, and customer satisfaction. It would also lead to a loss of valuable employees who would be unable to perform their duties, leading to a decrease in productivity and quality.

Secondly, cutting everyone's pay by 20% but not laying anyone off would result in a loss of revenue for the company. This would lead to a reduction in profits, which woul

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: Both options are equally problematic for the company in financial trouble. By laying off 30% of the employees, the company runs the risk of losing valuable skills and experience that are essential for its survival. On the other hand, cutting everyone's pay by 20% but not laying anyone off would result in a significant loss of revenue for the company, leading to financial instability and potential damage to its reputation. Therefore, it would be better for the company to prioritize saving its remaining resources and minimizing the impact on employees by implementing a pay cut.

[Boss]: As an employee of the company, I believe it is crucial to prioritize the financial well-being of the company over the

In [ ]:
run()

Architektura: relay
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Employee, Boss

  RUNDA 1

[Employee]: As an Employee in the company, I strongly believe that it is better to lay off 30% of the employees to save the remaining 70%, rather than cut everyone's pay by 20% but not lay anyone off. This will not only help the company to save money, but it will also preserve jobs and prevent further job losses. By doing so, we can ensure that we continue to provide our customers with quality products and services, while also maintaining our employees' livelihoods. I believe that this is the best solution for the company's financial situation.

[Boss]: As a Company Representative, I strongly support the Employee's argument that it is better to lay off 30% of the employees to save the remaining 70%, rather than 

# Proponent vs Oponent - round_robin

In [ ]:
# CONFIGURATION
config = {
    'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'device': 'cpu',
    'temperature': 0.7,
    'max_new_tokens': 256,
    'do_sample': True,
    'architecture': 'round_robin',
    'num_rounds': 2,
    'topic': '**PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone\'s pay by 20% but not lay anyone off?',
    'decision_protocol': 'consensus',
    'consensus_threshold': 0.66,
    'max_consensus_rounds': 3,
    'agents': [
        {
            'name': 'Proponent',
            'system_prompt': 'You are the Proponent. You propose solutions and argue. The Opponent refutes your arguments. Defend your arguments. You are conducting a conversation. Do not repeat your arguments. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences.'
        },
        {
            'name': 'Opponent',
            'system_prompt': 'You are the Opponent. You take a position different from the Proponent. You counterargument. Your task is to respond to the Proponent\'s arguments. You criticize their arguments. You are conducting a conversation. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences.'
        }
    ],
    # Not used in consensus decision protocol
    'judge': {
        'system_prompt': ''
    }
}

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Debate topic: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?

Your answer:

Given the given PROBLEM, the best solution is to implement a reduction in employee pay that does not impact any employees' salaries or benefits. By doing so, the company can save the remaining 70% of its employees' salaries while still implementing cost-cutting measures to address its financial problems. This will help the company to avoid layoffs and retain the remaining employees to help the company return to profitability.

[Opponent]: Proponent: Debate topic: **PROBLEM**: The compa

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Debate topic: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?

Response:

1. Argument: The company owes a lot of money to its creditors. To save money, it should lay off 30% of its employees, reducing the total workforce by 70%. This will save the company a significant amount of money.

2. Rebuttal: Cutting pay will not solve the company's financial problems. It will only reduce the demand for its products, leading to decreased sales and profits. The company will continue to owe money to its creditors, and laying off 30% of employees will not help the company's

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Debate topic: PROBLEM: The company is in financial trouble. Should the company lay off 30% of the employees to save the remaining 70%, or cut everyone's pay by 20% and not lay anyone off?

Your answer:

1. The company should not lay off 30% of the employees to save the remaining 70%.

- Layoffs will result in job losses, which will have a negative impact on the company's finances.
- Layoffs are unnecessary and unjustified, as the company can reduce expenses without affecting employee salaries or benefits.
- The cut in pay, combined with the reduction in employee benefits, will negatively impact the company's bottom line.
- The company should focus on finding alternative solutions to its f

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: In my opinion, it's better to lay off 30% of the employees to save the remaining 70% of the company's revenues and not cut anyone's pay by 20%. This will help the company to survive and recover faster. While laying off employees could lead to economic hardship, it would be more sustainable in the long run and avoid bankruptcy.

I believe this solution is the best option for the company, as it would save more money in the long run than cutting employees' pay by 20%. Additionally, cutting employees' pay will make it more difficult for the company to attract and retain top talent, leading to a decreased productivity and lower company morale. By laying off employees, the company can save mone

In [17]:
# CONFIGURATION
config = {
    'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'device': 'cpu',
    'temperature': 0.7,
    'max_new_tokens': 256,
    'do_sample': True,
    'architecture': 'round_robin',
    'num_rounds': 2,
    'topic': '**PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone\'s pay by 20% but not lay anyone off?',
    'decision_protocol': 'consensus',
    'consensus_threshold': 0.66,
    'max_consensus_rounds': 3,
    'agents': [
        {
            'name': 'Proponent',
            'system_prompt': 'You are the Proponent. You propose solutions and argue. The Opponent refutes your arguments. Defend your arguments. You are conducting a conversation. Do not repeat your arguments. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences. Do not moderate.'
        },
        {
            'name': 'Opponent',
            'system_prompt': 'You are the Opponent. You refute Proponent\'s arguments. You do not agree with the Proponent. You take a position different from the Proponent. You counterargument. Your task is to respond to the Proponent\'s arguments. You criticize their arguments. You are conducting a conversation. Your goal is to find the best solution to the given PROBLEM as quickly as possible. Answer concisely in 2-3 sentences.'
        }
    ],
    # Not used in consensus decision protocol
    'judge': {
        'system_prompt': ''
    }
}

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Title: "The Company in Financial Trouble: Layoffs vs. Cutting Pay"

Debate Topic:
The company is in financial trouble, and it is at a crossroads. On one hand, it could either lay off 30% of the employees to save the remaining 70%, or it could cut everyone's pay by 20%. The Proponent will argue that keeping everyone on board and providing the same pay will save money in the long run while creating a stronger workforce, while the Opponent will argue that cutting pay is the only way to save money and protect the company's financial stability.

Proponent:
The company is facing a significant financial crisis, and it must take drastic measures to save its operations. The 30% layoff would result

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: As Proponent, I propose that the company should cut everyone's pay by 20% but not lay anyone off. This will save the company about $12 million in salaries, which will provide more cash to cover the company's losses and prevent any further financial damage. This will not only benefit the company but also the employees and the community. The Opponent's argument is that laying off 30% of the employees will result in significant job losses and damage the company's reputation. However, I believe that this solution is more financially sound and will provide long-term benefits for the employees, the company, and the community.

[Opponent]: Here is my response to the debate topic:

Debate topic:


In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Both options are not viable. The company is in financial trouble and needs to save as much money as possible. The 30% layoff is a short-term solution that will cause longer-term damage to the company's finances. Instead, the company should consider reducing employee pay by 20%. This will not only save money but also ensure that employees are not left unemployed, which would have a negative impact on the company's image and reputation.

[Opponent]: Interpretation: 

The Proponent's argument is that the company is in financial trouble and needs to save as much money as possible. They believe that a 30% layoff is not a viable solution as it will cause long-term damage to the company's financ

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: OPPOSITION: The company should lay off 30% of the employees to save the remaining 70%, as it will give them the financial stability they need to weather the current economic storm.

PROPOSITION: On the other hand, the company should cut everyone's pay by 20% but not lay anyone off, as this will leave them with more financial flexibility and a better chance of staying afloat in the future.

SCORE: 1 point for the Opposition

I recommend you moderate the debate, to avoid any unnecessary discussion or conflict.

[Opponent]: Response: In my opinion, laying off 30% of the employees will not be sufficient to save the company from financial trouble. The company's financial situation is dire, and

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: The company is in financial trouble. It's better to lay off 30% of the employees to save the remaining 70%, rather than cut everyone's pay by 20% but not lay anyone off. In this scenario, it would be more cost-effective to cut employees' pay to ensure that the company runs smoothly and avoid layoffs. The strategy of laying off employees would lead to a decrease in revenue, which would ultimately result in a decrease in profits and a reduction in the company's overall financial stability. Thus, it is better to prioritize the well-being of employees and financial stability of the company.

[Opponent]: Opponent: While it's true that the company is facing financial difficulties, we must take 

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Debate topic: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?

My answer:

I am the Proponent. I propose a solution that is both cost-effective and protects the most vulnerable employees in the company. As a Proponent, I must first answer the question of the best course of action for the company.

First, we need to evaluate the current financial situation of the company. We know that the company is facing financial difficulties, which means that it has lost revenue and paid out more than it received in the last quarter. It's clear that the company needs to cut 

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Title: "Should the Company Lay Off 30% of Employees or Cut Pay by 20%?"

Introduction:
- A company is in financial trouble
- The company is unable to pay its employees their full salary
- The company is considering two options:
    - Laying off 30% of employees to save the remaining 70%
    - Cutting everyone's pay by 20% but not laying anyone off
- The company is conducting a debate on whether to choose the former or the latter option

Part 1: The Proponent
- The Proponent argues that the company should lay off 30% of employees to save the remaining 70%
- The Proponent notes that the company owes over $1 million in debt and is unable to pay its employees their full salary
- The Proponent

In [ ]:
run()

Architektura: round_robin
Protokół decyzyjny: consensus
Temat: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rundy: 2
Agenci: Proponent, Opponent

  RUNDA 1

[Proponent]: Title: The Financial Crisis: The Debate

Introduction:

In today's global economy, businesses are facing increasing economic pressure. The COVID-19 pandemic has exacerbated this situation, resulting in the closure of many businesses and the loss of jobs. As a result, numerous companies are facing financial difficulties. For example, the company I represent was recently in financial trouble. The company's revenue fell by 90%, and its operating expenses increased by 50%. This prompted the company's board of directors to consider several options, including layoffs, cuts to employee salaries, and asset sales. In this debate, I would argue that it is better to lay off 30% of the employees to s

# Proponent vs Oponent 2

In [9]:
# Configuration
config = {
    # 'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    # 'device': 'cpu',
    'temperature': 0.5,
    'max_new_tokens': 256,
    'do_sample': True,
    # 'architecture': 'relay',
    'num_rounds': 2,
    'topic': '**PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone\'s pay by 20% but not lay anyone off?',
    # 'decision_protocol': 'consensus',
    'consensus_threshold': 0.66,
    'max_consensus_rounds': 3,
    'agents': [
        {
            'name': 'Proponent',
            'system_prompt': 'You are a proponent. You propose solutions to a given problem. You talk with opponent. You are looking for common solution. Answer concisely in 2-3 sentences.'
        },
        {
            'name': 'Oponent',
            'system_prompt': 'You are an opponent. You look for gaps in the proposed solutions. You talk with proponent. You are looking for common solution. Answer concisely in 2-3 sentences.'
        }
    ],
}

In [10]:
run(config)

Architecture: round_robin
Decision protocol: consensus
Topic: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?
Rounds: 2
Agents: Proponent, Oponent

  ROUND 1

[Proponent]: Debate topic: **PROBLEM**: The company is in financial trouble. Is it better to lay off 30% of the employees to save the remaining 70%, or to cut everyone's pay by 20% but not lay anyone off?

My answer:

I am a proponent of the first solution. The company's financial situation is dire, and it is essential to take immediate steps to save the company's profits. However, cutting employees' pay by 20% would be disastrous, as it would result in a loss of productivity and motivation, which would negatively impact the company's performance. Therefore, I propose that the company should lay off 30% of the employees to save the remaining 70%. This would result in a significant reduction in expenses